In [1]:
# use kernel mira-env

from pathlib import Path
import scanpy as sc
import anndata as ad
import pandas as pd
import sys
import os
import numpy as np
import random
import scipy as sp
from matplotlib.pyplot import rc_context
from matplotlib import rcParams
from matplotlib import pyplot as plt
import pickle as pkl
import time
import warnings
import igraph
import h5py

from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import pdist, squareform

from io import BytesIO
import numpy as np
from scipy.sparse import coo_matrix
from scipy.io import mmwrite

In [2]:
cd4_ad = ad.read('data/published_data/cd4tcells_final.h5ad')

In [3]:
cd4_ad

AnnData object with n_obs × n_vars = 145704 × 33075
    obs: 'sample_id', 'Patient', 'Disease', 'Site', 'Treatment', 'Disease_duration', 'Inflammation', 'Age', 'Gender', 'Ethnicity', 'Inflammation_score', 'Ileum_vs_Colon', 'LibraryType', 'CellsLoaded', 'Match', 'Batch', 'doublet_scores', 'predicted_doublets', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rp', 'pct_counts_rp', 'total_counts_hb', 'pct_counts_hb', 'total_counts_ig', 'pct_counts_ig', 'S_score', 'G2M_score', 'phase', 'cellbarcode', 'diff', 'final_analysis', 'minor', 'major', 'sub_bucket', 'bucket', 'Remission_status'
    var: 'feature_types', 'gene_id', 'hb', 'ig', 'rp', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'gene_symbol'
    uns: 'final_analysis_colors', 'log1p', 'sample_id_colors'
    obsm: 'X_harmony', 'X_pca', 'X_umap'

In [4]:
cd4_ad.obs

,sample_id,Patient,Disease,Site,Treatment,Disease_duration,Inflammation,Age,Gender,Ethnicity,...,G2M_score,phase,cellbarcode,diff,final_analysis,minor,major,sub_bucket,bucket,Remission_status
AAACGAACAGCATTGT-1-CID003352-2,CID003352-2,UC1,UC,Rectum,Post,29.0,Inflamed,26,M,Caucasian,...,-0.177739,G1,AAACGAACAGCATTGT-1-CID003352-2,8.0,Th22,CD4_T_helper,CD4_T,CD4_cytotoxic_lo_T_cell,CD4_cytotoxic_lo_T_cell,Non_Remission
AAACGAACAGCGTTTA-1-CID003352-2,CID003352-2,UC1,UC,Rectum,Post,29.0,Inflamed,26,M,Caucasian,...,-0.255245,G1,AAACGAACAGCGTTTA-1-CID003352-2,8.0,CD4 FOSpos T,CD4_T_memory,CD4_T,CD4_cytotoxic_lo_T_cell,CD4_cytotoxic_lo_T_cell,Non_Remission
AAAGAACGTTGCCAAT-1-CID003352-2,CID003352-2,UC1,UC,Rectum,Post,29.0,Inflamed,26,M,Caucasian,...,-0.248834,G1,AAAGAACGTTGCCAAT-1-CID003352-2,8.0,Th17,CD4_T_helper,CD4_T,CD4_cytotoxic_lo_T_cell,CD4_cytotoxic_lo_T_cell,Non_Remission
AAAGGGCTCACTCCGT-1-CID003352-2,CID003352-2,UC1,UC,Rectum,Post,29.0,Inflamed,26,M,Caucasian,...,-0.400350,G1,AAAGGGCTCACTCCGT-1-CID003352-2,8.0,Th17,CD4_T_helper,CD4_T,CD4_cytotoxic_lo_T_cell,CD4_cytotoxic_lo_T_cell,Non_Remission
AAAGTCCGTATTAAGG-1-CID003352-2,CID003352-2,UC1,UC,Rectum,Post,29.0,Inflamed,26,M,Caucasian,...,-0.091492,G1,AAAGTCCGTATTAAGG-1-CID003352-2,8.0,Th22,CD4_T_helper,CD4_T,CD4_cytotoxic_lo_T_cell,CD4_cytotoxic_lo_T_cell,Non_Remission
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGATCTCGGACTGC-1-CID006584-1,CID006584-1,UC21,UC,Rectum,Pre,3.0,Inflamed,54,F,Caucasian,...,-0.067599,G1,TTTGATCTCGGACTGC-1-CID006584-1,NaN,Th1 17 GZMAhi,CD4_T_helper,CD4_T,CD4_cytotoxic_lo_T_cell,CD4_cytotoxic_lo_T_cell,Not_avail
TTTGGAGTCACGGAGA-1-CID006584-1,CID006584-1,UC21,UC,Rectum,Pre,3.0,Inflamed,54,F,Caucasian,...,-0.226107,G1,TTTGGAGTCACGGAGA-1-CID006584-1,NaN,CD4 KLF2int T,CD4_T_memory,CD4_T,CD4_cytotoxic_lo_T_cell,CD4_cytotoxic_lo_T_cell,Not_avail
TTTGGTTCAGCGTTGC-1-CID006584-1,CID006584-1,UC21,UC,Rectum,Pre,3.0,Inflamed,54,F,Caucasian,...,-0.142191,G1,TTTGGTTCAGCGTTGC-1-CID006584-1,NaN,CD4 FOSpos T,CD4_T_memory,CD4_T,CD4_cytotoxic_lo_T_cell,CD4_cytotoxic_lo_T_cell,Not_avail
TTTGTTGCAAGCGCTC-1-CID006584-1,CID006584-1,UC21,UC,Rectum,Pre,3.0,Inflamed,54,F,Caucasian,...,-0.191142,G1,TTTGTTGCAAGCGCTC-1-CID006584-1,NaN,Th17,CD4_T_helper,CD4_T,CD4_cytotoxic_lo_T_cell,CD4_cytotoxic_lo_T_cell,Not_avail


In [5]:
cd4_ad.var

,feature_types,gene_id,hb,ig,rp,mt,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,gene_symbol
MIR1302-2HG,Gene Expression,ENSG00000243485,False,False,False,False,7,0.000004,99.999586,7.0,MIR1302-2HG
FAM138A,Gene Expression,ENSG00000237613,False,False,False,False,0,0.000000,100.000000,0.0,FAM138A
OR4F5,Gene Expression,ENSG00000186092,False,False,False,False,10,0.000006,99.999408,10.0,OR4F5
AL627309.1,Gene Expression,ENSG00000238009,False,False,False,False,2267,0.001351,99.865826,2283.0,AL627309.1
AL627309.3,Gene Expression,ENSG00000239945,False,False,False,False,93,0.000055,99.994496,93.0,AL627309.3
...,...,...,...,...,...,...,...,...,...,...,...
AC004556.1,Gene Expression,ENSG00000276345,False,False,False,False,14984,0.010305,99.113160,17412.0,AC004556.1
AC233755.2,Gene Expression,ENSG00000277856,False,False,False,False,5730,0.014294,99.660865,24151.0,AC233755.2
AC233755.1,Gene Expression,ENSG00000275063,False,False,False,False,6883,0.010157,99.592624,17162.0,AC233755.1
AC240274.1,Gene Expression,ENSG00000271254,False,False,False,False,19584,0.012364,98.840905,20890.0,AC240274.1


In [7]:
mmwrite("./data/published_data/thomas_cd4_ad.mtx", cd4_ad.X)

In [8]:
cd4_ad.var.to_csv('./data/published_data/thomas_cd4_ad_genes.csv')

In [9]:
cd4_ad.obs.to_csv('./data/published_data/thomas_cd4_ad_meta.csv', index=False)

# Other celltypes

In [14]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/084_Taurus_IBD/data/all_data/ilc_final.h5ad')
#dat_ad.obs.to_csv('./data/published_data/thomas_ilc_ad_meta.csv', index=False)
mmwrite("./data/published_data/thomas_ilc_ad.mtx", dat_ad.X)
dat_ad.var.to_csv('./data/published_data/thomas_ilc_ad_genes.csv')

In [15]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/084_Taurus_IBD/data/all_data/myeloid_final.h5ad')
#dat_ad.obs.to_csv('./data/published_data/thomas_myeloid_ad_meta.csv', index=False)
mmwrite("./data/published_data/thomas_myeloid_ad.mtx", dat_ad.X)
dat_ad.var.to_csv('./data/published_data/thomas_myeloid_ad_genes.csv')

In [16]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/084_Taurus_IBD/data/all_data/bcells_final.h5ad')
#dat_ad.obs.to_csv('./data/published_data/thomas_bcells_ad_meta.csv', index=False)
mmwrite("./data/published_data/thomas_bcells_ad.mtx", dat_ad.X)
dat_ad.var.to_csv('./data/published_data/thomas_bcells_ad_genes.csv')

In [17]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/084_Taurus_IBD/data/all_data/cd8tcells_final.h5ad')
#dat_ad.obs.to_csv('./data/published_data/thomas_cd8tcells_ad_meta.csv', index=False)
mmwrite("./data/published_data/thomas_cd8tcells_ad.mtx", dat_ad.X)
dat_ad.var.to_csv('./data/published_data/thomas_cd8tcells_ad_genes.csv')

In [18]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/084_Taurus_IBD/data/all_data/ilealepi_final.h5ad')
#dat_ad.obs.to_csv('./data/published_data/thomas_ilealepi_ad_meta.csv', index=False)
mmwrite("./data/published_data/thomas_ilealepi_ad.mtx", dat_ad.X)
dat_ad.var.to_csv('./data/published_data/thomas_ilealepi_ad_genes.csv')

In [19]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/084_Taurus_IBD/data/all_data/vasc_final.h5ad')
#dat_ad.obs.to_csv('./data/published_data/thomas_vasc_ad_meta.csv', index=False)
mmwrite("./data/published_data/thomas_vasc_ad.mtx", dat_ad.X)
dat_ad.var.to_csv('./data/published_data/thomas_vasc_ad_genes.csv')

In [20]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/084_Taurus_IBD/data/all_data/fibperi_final.h5ad')
#dat_ad.obs.to_csv('./data/published_data/thomas_fibperi_ad_meta.csv', index=False)
mmwrite("./data/published_data/thomas_fibperi_ad.mtx", dat_ad.X)
dat_ad.var.to_csv('./data/published_data/thomas_fibperi_ad_genes.csv')

In [21]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/084_Taurus_IBD/data/all_data/plasmacells_final.h5ad')
#dat_ad.obs.to_csv('./data/published_data/thomas_plasmacells_ad_meta.csv', index=False)
mmwrite("./data/published_data/thomas_plasmacells_ad.mtx", dat_ad.X)
dat_ad.var.to_csv('./data/published_data/thomas_plasmacells_ad_genes.csv')

In [22]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/084_Taurus_IBD/data/all_data/epicolonic_final.h5ad')
#dat_ad.obs.to_csv('./data/published_data/thomas_epicolonic_ad_meta.csv', index=False)
mmwrite("./data/published_data/thomas_epicolonic_ad.mtx", dat_ad.X)
dat_ad.var.to_csv('./data/published_data/thomas_epicolonic_ad_genes.csv')

# Teichmann Pediatric Crohn

In [23]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/063_Teichman_colon/pediatric_RAWCOUNTS_cellxgene_c.h5ad')
mmwrite("../../DATA_scRNAseq/Analysis_of_previously_published_data/063_Teichman_colon/pediatric_RAWCOUNTS_cellxgene_c.mtx", dat_ad.X)
dat_ad.var.to_csv('../../DATA_scRNAseq/Analysis_of_previously_published_data/063_Teichman_colon/pediatric_RAWCOUNTS_cellxgene_c.csv')

In [2]:
dat_ad = ad.read('../../DATA_scRNAseq/Analysis_of_previously_published_data/063_Teichman_colon/pediatric_RAWCOUNTS_cellxgene_c.h5ad')


In [3]:
dat_ad.obs.to_csv('../../DATA_scRNAseq/Analysis_of_previously_published_data/063_Teichman_colon/pediatric_RAWCOUNTS_cellxgene_c_ad_meta.csv', index=False)


# CDN for NicheNet

In [2]:
dat_ad = ad.read('data/cdn/adata_tissue_sub_100k.h5ad')

In [3]:
mmwrite("data/cdn/adata_tissue_sub100k.mtx", dat_ad.X)
dat_ad.var.to_csv('data/cdn/adata_tissue_sub_genes100k.csv')

In [4]:
dat_ad.obs.to_csv('data/cdn/adata_tissue_sub_md100k.csv', index=False)


/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `ls -F --color ()'
